In [69]:
import pandas as pd

df = pd.read_csv('alphatrade_phase1.csv')
df.head()

,Date,Close,High,Low,Open,Volume,SMA_20,SMA_50,EMA_20,EMA_50,Daily_Return,Volatility,RSI,MACD,Signal_Line,BB_Middle,BB_Upper,BB_Lower,Signal
0,2020-01-02,72.333855,72.394063,71.091161,71.344032,135480400,NaN,NaN,72.333855,72.333855,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,0
1,2020-01-03,71.630630,72.389250,71.406659,71.563198,146322800,NaN,NaN,72.266881,72.306277,-0.009722,NaN,NaN,-0.056098,-0.011220,NaN,NaN,NaN,0
2,2020-01-06,72.201401,72.239935,70.503539,70.754006,118387200,NaN,NaN,72.260645,72.302164,0.007968,NaN,NaN,-0.053878,-0.019751,NaN,NaN,NaN,0
3,2020-01-07,71.861855,72.466338,71.642697,72.211056,108872000,NaN,NaN,72.222665,72.284897,-0.004703,NaN,NaN,-0.078611,-0.031523,NaN,NaN,NaN,0
4,2020-01-08,73.017815,73.318854,71.565599,71.565599,132079200,NaN,NaN,72.298393,72.313639,0.016086,NaN,NaN,-0.004880,-0.026195,NaN,NaN,NaN,0


In [70]:
#Step 1: Inspect the loaded dataset
print(df.shape)
print(df.columns)
df.info()

(1508, 19)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50',
       'EMA_20', 'EMA_50', 'Daily_Return', 'Volatility', 'RSI', 'MACD',
       'Signal_Line', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Signal'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 1508 entries, 0 to 1507
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          1508 non-null   str    
 1   Close         1508 non-null   float64
 2   High          1508 non-null   float64
 3   Low           1508 non-null   float64
 4   Open          1508 non-null   float64
 5   Volume        1508 non-null   int64  
 6   SMA_20        1489 non-null   float64
 7   SMA_50        1459 non-null   float64
 8   EMA_20        1508 non-null   float64
 9   EMA_50        1508 non-null   float64
 10  Daily_Return  1507 non-null   float64
 11  Volatility    1488 non-null   float64
 12  RSI           1495 non-null   float64
 13  MACD 

In [71]:
#Step 2: Check for missing values
df.isnull().sum()
'Many indicators like SMA, RSI, and Bollinger Bands create missing values at the beginning because they need historical data.'

'Many indicators like SMA, RSI, and Bollinger Bands create missing values at the beginning because they need historical data.'

In [72]:
#Step 3: Remove missing values
df = df.dropna()

#Verify:

print(df.isnull().sum())
print(df.shape)

Date            0
Close           0
High            0
Low             0
Open            0
Volume          0
SMA_20          0
SMA_50          0
EMA_20          0
EMA_50          0
Daily_Return    0
Volatility      0
RSI             0
MACD            0
Signal_Line     0
BB_Middle       0
BB_Upper        0
BB_Lower        0
Signal          0
dtype: int64
(1459, 19)


In [73]:
print(df.shape)
print(df.columns)

df = df.dropna()

df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

df = df[:-1]

print(df.shape)
print(df['Target'].value_counts())

(1459, 19)
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA_20', 'SMA_50',
       'EMA_20', 'EMA_50', 'Daily_Return', 'Volatility', 'RSI', 'MACD',
       'Signal_Line', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Signal'],
      dtype='str')
(1458, 20)
Target
1    778
0    680
Name: count, dtype: int64


In [74]:
# Features and creating X & Y
features = [
    'Close',
    'Volume',
    'SMA_20',
    'SMA_50',
    'EMA_20',
    'EMA_50',
    'Daily_Return',
    'Volatility',
    'RSI',
    'MACD',
    'Signal_Line'
]

X = df[features]
y = df['Target']

print(X.shape)
print(y.shape)

(1458, 11)
(1458,)


In [75]:
# Scaling dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(1166, 11)
(292, 11)


In [76]:
#Train the model Logistic Regression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [77]:
'Train model Random Forest'
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))

print(confusion_matrix(y_test, rf_pred))

Accuracy: 0.5273972602739726
              precision    recall  f1-score   support

           0       0.46      0.42      0.44       129
           1       0.57      0.61      0.59       163

    accuracy                           0.53       292
   macro avg       0.52      0.52      0.52       292
weighted avg       0.52      0.53      0.52       292

[[ 54  75]
 [ 63 100]]


In [78]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

         Feature  Importance
6   Daily_Return    0.111647
1         Volume    0.105207
7     Volatility    0.103941
8            RSI    0.102538
9           MACD    0.096365
0          Close    0.090838
10   Signal_Line    0.090484
4         EMA_20    0.075270
2         SMA_20    0.074927
3         SMA_50    0.074600
5         EMA_50    0.074183


In [79]:
'Model xgboost'
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, xgb_pred))

print(classification_report(y_test, xgb_pred))

print(confusion_matrix(y_test, xgb_pred))

Accuracy: 0.5068493150684932
              precision    recall  f1-score   support

           0       0.44      0.41      0.42       129
           1       0.56      0.58      0.57       163

    accuracy                           0.51       292
   macro avg       0.50      0.50      0.50       292
weighted avg       0.50      0.51      0.50       292

[[53 76]
 [68 95]]


In [80]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

         Feature  Importance
6   Daily_Return    0.100359
0          Close    0.100320
3         SMA_50    0.098194
7     Volatility    0.094358
5         EMA_50    0.091797
10   Signal_Line    0.091048
1         Volume    0.088829
9           MACD    0.086906
8            RSI    0.085425
4         EMA_20    0.082449
2         SMA_20    0.080315


In [81]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [82]:
# Retrain Logistic Regression:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_model = LogisticRegression()

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

print("Logistic Regression:", accuracy_score(y_test, lr_pred))

Logistic Regression: 0.5376712328767124


In [83]:
# Retrain Random Forest:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest:", accuracy_score(y_test, rf_pred))

Random Forest: 0.5273972602739726


In [84]:
# Retrain XGBoost:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost:", accuracy_score(y_test, xgb_pred))

XGBoost: 0.5068493150684932


In [85]:
print(X_train.shape)
print(X_test.shape)

(1166, 11)
(292, 11)
